# 04 — Disección instrumentada de SAM 2.1 y DINOv3

**Solo etapa 4; ejecución por el usuario exclusivamente en Colab con GPU.** No ejecutar modelos en la PC, tampoco en CPU. No entrenamiento, probe, prompts automáticos, métricas de calidad, reservados ni demo.

Reutiliza las etapas 1–3: entorno, SAM tiny, datos y estadísticas train por régimen, transforms, geometría y persistencia verificada. DINOv3 ViT-L/16 SAT-493M congelado, revisión autorizada `f692fa42da72c6797b67cd73494a168d1120d3ee`. Conserva la salvedad histórica de licencia de datos.

Fuentes: bases del curso, sección A y política IA; resumen WAIMLAP, metodología y evaluación; hoja de ruta, §§2.4 y etapa 4. Instrucciones completas: `docs/project/validacion_etapa4.md` en el paquete.


## Hipótesis y protocolo previo

H4a: el encoder de imagen dominará la aritmética cubierta de SAM; sus módulos de vídeo siguen teniendo parámetros aunque no se ejecuten. H4b: atención global de DINOv3 permitirá sensibilidad distante no uniforme. H4c: FLOPs parciales y latencia no serán equivalentes por tráfico, operaciones sin fórmula y kernels.

Dos chips train por régimen, primeros IDs lexicográficos, sin etiquetas. RGB B4/B3/B2 y límites radiométricos de etapa 2 sin reajustarlos; SAM 1024 y DINO 512. Batch 1, float32, sin autocast/TF32, semilla 42; 5 warmups y 20 repeticiones sincronizadas. Puntos diagnósticos espaciales fijos (256,256)+ y (32,32)−, sin interpretación semántica ni calidad de segmentación. No cambies estas decisiones después de ver las imágenes.

Parámetros por propiedad exclusiva con residual; FLOPs PyTorch con 2 operaciones por multiply-add, cobertura parcial listada dinámicamente. Features Hiera de primer/medio/último bloque y DINO capas 0/11/23 más salida normalizada. CLS=0, registros después, parches restantes en orden de filas. Visualización de canales 0/1/2; sin PCA ni proyección aprendida. ERF: gradiente absoluto RGB respecto a entrada normalizada, canal 0 espacial central, masa unitaria; no campo receptivo teórico.


## 1. Paquete y entorno

En la PC: `python scripts/export_stage4_bundle.py`. Sube el ZIP de fuentes a Drive, abre este notebook en Colab y selecciona GPU. Copia su SHA-256 debajo. Usa la raíz persistente histórica con `raw/`, `stage2/` y manifiestos/preprocesamiento por régimen. La revisión gated requiere tu token autorizado de Hugging Face si no está en caché; usa el secreto `HF_TOKEN` de Colab sin imprimirlo.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage4')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage4.zip')
EXPECTED_BUNDLE_SHA256 = ''  # Copiar salida de export_stage4_bundle.py.
assert len(EXPECTED_BUNDLE_SHA256) == 64, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
# Token opcional para descarga autorizada, nunca imprimirlo ni incluirlo en informes.
from google.colab import userdata
try:
    child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print('HF_TOKEN no configurado; se usara la cache autorizada si existe.')
LOG_DIR = Path(PERSIST_ROOT) / 'stage4/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def command(args, log_name):
    with (LOG_DIR / log_name).open('w', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; conservar salida y log {log_name}')
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')


## 2. Preparar y medir en Colab

Ejecuta `prepare` una vez. Carga SAM y DINO secuencialmente para limitar VRAM; no cambia modelo, precisión o tamaño ante OOM. Un fallo guarda diagnóstico y generaciones completas; conserva el ZIP, corrige el recurso en Colab y usa `prepare` para un run nuevo. Las mediciones se hacen sobre el primer train de cada régimen; las features/ERF sobre ambos.

El flujo SAM tensorial se contrasta con el predictor reutilizado de etapa 3. Los tiempos excluyen descarga/carga de pesos, TIFF, transforms CPU, H2D/D2H, hooks, gradientes, guardado y figuras. SAM incluye encoder, proyecciones conv_s0/s1, no_mem, prompts, dense PE, decoder y logits a 512. DINO incluye embeddings, RoPE, 24 capas y norma final. El replay aislado de bloques no es aditivo y declara su baseline de memoria. El flujo automático completo aún no existe.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_stage4.py'), 'prepare', '--root', PERSIST_ROOT], 'prepare.log')


## 3. Inspeccionar evidencia real

Revisa tablas y figuras. Los colores de cada canal usan min/max del panel; el ERF usa color logarítmico y overlay sobre RGB train. Los arrays crudos y los rangos se guardan. No interpretes canales como probabilidad de agua. Las columnas FLOPs nulas indican módulos no llamados; los ceros del contador no acreditan costo real cero.


In [ ]:
from IPython.display import display, Image, HTML
import csv, html
EVIDENCE = Path(PERSIST_ROOT) / 'stage4/evidence'
RUNS = sorted(p for p in EVIDENCE.iterdir() if p.is_dir() and (p / 'validation.json').is_file())
assert RUNS, 'No hay prepare completo; conserva logs y failure.json.'
RUN_DIR = RUNS[-1]
REPORT = json.loads((RUN_DIR / 'validation.json').read_text())
print('run_id:', REPORT['run_id'])
print(json.dumps(REPORT['interpretation'], indent=2, ensure_ascii=False))
for table in sorted(RUN_DIR.glob('*.csv')):
    rows = list(csv.reader(table.open(encoding='utf-8', newline='')))
    print(table.name)
    display(HTML('<table>' + ''.join('<tr>' + ''.join('<td>' + html.escape(v) + '</td>' for v in row) + '</tr>' for row in rows) + '</table>'))
for figure in sorted(RUN_DIR.glob('*_0_figure.png')):
    print(figure.name)
    display(Image(filename=str(figure)))
print('Todos los arrays, metadatos y figuras:', RUN_DIR.with_suffix('.zip'))


## 4. Recuperación offline tras eliminar el runtime

Guarda el notebook **con salidas**, desconecta y elimina el runtime desde Colab, vuelve a seleccionar GPU y ejecuta solo la celda de entorno de la sección 1 y la celda `recover` siguiente. No repitas `prepare`: crearía otro run. Mantén ZIP, SHA, config y raíz idénticos. Se reconstruye el entorno, se recargan los pesos en caché persistente con `local_files_only=True`, se verifican todas las generaciones y se repiten features/ERF en los mismos train con atol=1e-5/rtol=1e-4. No se vuelven a medir latencias para sustituir las originales. Un boot ID distinto es obligatorio. No ejecutar las celdas prepare/inspección en esta segunda sesión; sus salidas previas deben conservarse.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_stage4.py'), 'recover', '--root', PERSIST_ROOT], 'recover.log')


## 5. Entrega para revisión y parada

Comparte el notebook descargado con salidas y errores/orden conservados; el ZIP `stage4/evidence/<run_id>.zip`; `validation.json`, `recovery_validation.json`, `config.json`, `environment.json`; y los logs de `stage4/notebook_logs/` si hubo problemas. No compartas tokens ni pesos. Si falla un paso, comparte su evidencia y no avances de etapa.

El asistente verificará hashes, revisiones, train por régimen, parámetros/FLOPs, tiempos/memoria, hooks, pesos, recuperación y figuras; contrastará H4a–c con las medidas y actualizará cuaderno/estado. El marcador `complete` acredita cómputo, **no cierre académico de etapa**. La etapa 4 permanece en curso hasta esa revisión. Detener aquí; no ejecutar etapas 5–9 ni demo.


In [ ]:
RECOVERY_RUNS = sorted(p for p in (Path(PERSIST_ROOT) / 'stage4/evidence').iterdir() if p.is_dir() and (p / 'recovery_validation.json').is_file())
assert RECOVERY_RUNS
recovery = json.loads((RECOVERY_RUNS[-1] / 'recovery_validation.json').read_text())
print(json.dumps({'run_id': recovery['run_id'], 'status': recovery['status'], 'different_runtime_boot': recovery['different_runtime_boot'], 'closure': recovery['closure'], 'evidence_zip': str(RECOVERY_RUNS[-1].with_suffix('.zip'))}, indent=2))
